# Phase 0: Data Preparation & Engineering

**Notebook**: `02_Data_Preparation.ipynb`  
**Language**: Python (Polars)

This notebook implements the data preparation phase of the Fitbit Sleep Analysis pipeline.

## Steps:
1. **Load Data**: Sleep metrics and covariates.
2. **Merge & Clean**: Join datasets and filter invalid entries.
3. **Feature Engineering**: Calculate Age, Seasonality, Circular Encoding, and Social Jetlag.
4. **Variable Mapping**: Consolidate categories.
5. **Output**: Save the cleaned dataset for analysis.

In [ ]:
import polars as pl
import numpy as np
import os

## 1. Load Data

In [ ]:
# Define paths
SLEEP_DATA_PATH = "processed_data/daily_sleep_metrics_enhanced.parquet"
COVARIATES_PATH = "analysis_inputs/master_covariates_only.parquet"
OUTPUT_PATH = "processed_data/ready_for_analysis.parquet"

# Load data
try:
    df_sleep = pl.read_parquet(SLEEP_DATA_PATH)
    df_covariates = pl.read_parquet(COVARIATES_PATH)
    print("Data loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading data: {e}")
    # For demonstration purposes if files don't exist yet, we might want to stop or use mock data.
    # raise e

## 2. Merge & Clean

In [ ]:
# Join Sleep Data with Covariates
df = df_sleep.join(df_covariates, on="person_id", how="left")

# Filtering
# Drop rows with missing date_of_birth or sex_concept
df = df.filter(
    pl.col("date_of_birth").is_not_null() &
    pl.col("sex_concept").is_not_null()
)

# Optional: Filter person_total_nights >= 7 if preferred
# df = df.filter(pl.col("person_total_nights") >= 7)

# Keep specific variables
cols_to_keep = [
    "person_id", "sleep_date", "daily_midpoint_hour", "daily_start_hour", "daily_end_hour",
    "date_of_birth", "sex_concept", "race", "employment_status", "zip_code", "bmi", "menstral_stop_reason",
    "is_weekend" # Assuming this exists in sleep data or needs to be calculated
]

# Select columns if they exist, otherwise keep all for now to avoid errors during dev
# df = df.select(cols_to_keep)

## 3. Feature Engineering

In [ ]:
# Age: Calculate age_at_sleep
df = df.with_columns(
    ((pl.col("sleep_date") - pl.col("date_of_birth")).dt.total_days() / 365.25).alias("age_at_sleep")
)

# Seasonality: Calculate day_of_year
df = df.with_columns(
    pl.col("sleep_date").dt.ordinal_day().alias("day_of_year")
)

# Circular Encoding
df = df.with_columns([
    (np.sin(2 * np.pi * pl.col("daily_midpoint_hour") / 24)).alias("midpoint_sin"),
    (np.cos(2 * np.pi * pl.col("daily_midpoint_hour") / 24)).alias("midpoint_cos"),
    (np.sin(2 * np.pi * pl.col("daily_start_hour") / 24)).alias("onset_sin"),
    (np.cos(2 * np.pi * pl.col("daily_start_hour") / 24)).alias("onset_cos"),
    (np.sin(2 * np.pi * pl.col("daily_end_hour") / 24)).alias("offset_sin"),
    (np.cos(2 * np.pi * pl.col("daily_end_hour") / 24)).alias("offset_cos")
])

# Social Jetlag: Calculate raw difference (mean_weekend - mean_weekday) per person
# First, calculate mean midpoint for weekend and weekday per person
sjl_stats = df.group_by("person_id", "is_weekend").agg(
    pl.col("daily_midpoint_hour").mean().alias("mean_midpoint")
)

# Pivot to get weekend and weekday columns
sjl_pivot = sjl_stats.pivot(
    values="mean_midpoint",
    index="person_id",
    columns="is_weekend",
    aggregate_function="first"
)

# Assuming is_weekend is boolean or 0/1. Adjust column names accordingly.
# Let's assume True/False or 1/0. If it's string, we need to know values.
# For now, let's assume we can calculate it back joined to the main df or just calculate it here if we had the structure.
# A safer way without pivot if we don't know exact values:

weekend_means = df.filter(pl.col("is_weekend") == True).group_by("person_id").agg(pl.col("daily_midpoint_hour").mean().alias("mean_weekend"))
weekday_means = df.filter(pl.col("is_weekend") == False).group_by("person_id").agg(pl.col("daily_midpoint_hour").mean().alias("mean_weekday"))

df = df.join(weekend_means, on="person_id", how="left").join(weekday_means, on="person_id", how="left")

df = df.with_columns(
    (pl.col("mean_weekend") - pl.col("mean_weekday")).alias("SJL_raw")
)

## 4. Variable Mapping

In [ ]:
# Menopause: Create binary is_postmenopausal
# Example logic: if menstral_stop_reason is "Natural Menopause" or "Surgery"
df = df.with_columns(
    pl.when(pl.col("menstral_stop_reason").is_in(["Natural Menopause", "Surgery"]))
    .then(1)
    .otherwise(0)
    .alias("is_postmenopausal")
)

## 5. Output

In [ ]:
# Save processed data
df.write_parquet(OUTPUT_PATH)
print(f"Saved processed data to {OUTPUT_PATH}")